In [1]:
import pandas as pd
import os
import pyarrow
from pathlib import Path
import sys




#PROJECT_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = Path.cwd().resolve().parents[0]
SILVER_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'silver'
BRONZE_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'bronze'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Patch Summary Dimension

In [ ]:

patches_desc_df = pd.read_csv(BRONZE_DATASETS_DIR / 'patches/scraped_patch_notes.csv')

#patches_desc_df.info()
#patches_desc_df.head(5)


patches_desc_df['patch_start_date'] = pd.to_datetime(patches_desc_df['patch_date'], format='%m/%d/%Y', errors='coerce')
patches_desc_df = patches_desc_df.sort_values(by='patch_start_date', ascending=True)
patches_desc_df['patch_end_date'] = patches_desc_df['patch_start_date'].shift(-1)

patches_desc_df.head(5)

patches_dimension_dir = SILVER_DATASETS_DIR / 'patches_dimension.parquet'
patches_desc_df[['patch_number', 'patch_start_date', 'patch_end_date', 'patch_url']].to_parquet(patches_dimension_dir, engine='pyarrow', index=False)

In [ ]:
##Champions Changes Dimension

patches = pd.read_parquet(SILVER_DATASETS_DIR/'patches_dimension.parquet', engine='pyarrow')

if not os.path.exists(SILVER_DATASETS_DIR / 'champions_changes_fact.parquet'):
    champions_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
else:
    champions_changes = pd.read_parquet(SILVER_DATASETS_DIR/'champions_changes_dimension.parquet', engine='pyarrow')

for patch in patches.itertuples():
    if patch.patch_number not in champions_changes['patch_number'].values:
        source_file = BRONZE_DATASETS_DIR / f'patches/patch_highlights_champion_changes_{patch.patch_number}.csv'
        if os.path.exists(source_file):
            source_file_data = pd.read_csv(source_file)
            new_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
            new_changes['patch_number'] = source_file_data['Patch Number']
            new_changes['champion_name'] = source_file_data['Champion']
            new_changes['change_type'] = source_file_data['Change Type']
            champions_changes = pd.concat([champions_changes, new_changes], ignore_index=True)
        else:
            print(f"File {source_file} does not exist.")

champions_changes.to_parquet(SILVER_DATASETS_DIR / 'champions_changes_dimension.parquet', engine='pyarrow', index=False)


## Pro Matches Hist

In [ ]:
oe_file_dir = BRONZE_DATASETS_DIR / 'oe' / '2018_LoL_esports_match_data_from_OraclesElixir.csv'

oe_df = pd.read_csv(oe_file_dir)
match_summary_columns = ['date','side','position','playername','champion','gamelength','result','kills','deaths','assists',
                        'teamkills','teamdeaths','doublekills','triplekills','quadrakills','pentakills','damagetochampions',
                        'dpm','damageshare','damagetakenperminute','damagemitigatedperminute','damagetotowers','total cs',
                        'killsat10','assistsat10','deathsat10','csat10','opp_killsat10','opp_assistsat10','opp_deathsat10','opp_csat10',
                        'killsat15','assistsat15','deathsat15','csat15','opp_killsat15','opp_assistsat15','opp_deathsat15','opp_csat15',
                        'killsat20','assistsat20','deathsat20','csat20','opp_killsat20','opp_assistsat20','opp_deathsat20','opp_csat20',
                        'killsat25','assistsat25','deathsat25','csat25','opp_killsat25','opp_assistsat25','opp_deathsat25','opp_csat25']
ban_list_columns = ['date', 'gameid', 'side', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5']

oe_df['date'] = pd.to_datetime(oe_df['date'], errors='coerce')
matches_summary_df = (oe_df.loc[:, match_summary_columns].rename(columns={'total cs': 'total_cs'}).copy().dropna(subset=['date','champion']))
matches_summary_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_summary.parquet', engine='pyarrow', index=False)

ban_list_df = (oe_df.loc[:, ban_list_columns].copy().dropna(subset=['date','gameid','ban1']).drop_duplicates(subset=['date','gameid','side'], keep='first'))
ban_list_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_ban_list.parquet', engine='pyarrow', index=False)


In [3]:
from matches import get_patch_dates, get_last_match_date
from datetime import datetime

##Get first and last patch dates from the patches dimension
first_available_patch_date, last_available_patch_date = get_patch_dates()

##Get last match date from the procceded matches
last_processed_match_date = get_last_match_date()

years_list = list(range(first_available_patch_date.year, datetime.now().year + 1))

print(f"First available patch date: {first_available_patch_date}")
print(f"Last available patch date: {last_available_patch_date}")
print(f"Last processed match date: {last_processed_match_date}")
print(f"Years list: {years_list}")
print(f"Current year: {datetime.now().year}")

ImportError: cannot import name 'get_last_match_date' from 'matches' (c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py)

In [2]:
from matches import upload_oe_matches

result = upload_oe_matches(rewrite=True)
print(f"Upload result: {result}")

Processed year 2019: 97500 rows, 81250 total matches, 80823 total bans.
Processed year 2020: 116964 rows, 178720 total matches, 177794 total bans.
Processed year 2021: 147624 rows, 301740 total matches, 300379 total bans.


c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:45: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)


Processed year 2022: 150312 rows, 427000 total matches, 423658 total bans.


c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:45: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)


Processed year 2023: 133272 rows, 538060 total matches, 525543 total bans.


...

Processed year 2024: 122304 rows, 639980 total matches, 621221 total bans.


c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:45: DtypeWarning: Columns (0: url, 1: split) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)


Processed year 2025: 120456 rows, 740360 total matches, 721176 total bans.


c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:45: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)


Processed year 2026: 71784 rows, 800180 total matches, 780187 total bans.
Upload result: True


In [4]:
# ban_list_columns = ['date', 'patch', 'gameid', 'side', 'ban']
# year = 2019
# source_file_dir = BRONZE_DATASETS_DIR / 'oe' / f'{year}_LoL_esports_match_data_from_OraclesElixir.csv'
# ban_list_df = pd.DataFrame(columns=ban_list_columns)
# oe_df = pd.read_csv(source_file_dir)
# oe_df['date'] = pd.to_datetime(oe_df['date'], errors='coerce')
# ban_list_df = pd.concat(
#     [
#         ban_list_df,
#         oe_df.melt(
#             id_vars=['date', 'patch', 'gameid', 'side'],
#             value_vars=['ban1', 'ban2', 'ban3', 'ban4', 'ban5'],
#             var_name='ban_number',
#             value_name='ban'
#         )
#         .drop(columns=['ban_number'])
#         .dropna(subset=['date', 'gameid', 'ban'])
#         .drop_duplicates(subset=['date','gameid','side','ban'], keep='first')
#     ],
#     ignore_index=True
# )
# ban_list_df.iloc[ban_list_df['gameid']=='ESPORTSTMNT01/1030526'].head(10)

x = pd.read_parquet(SILVER_DATASETS_DIR / 'pro_matches_ban_list.parquet', engine='pyarrow')
x.iloc[x['gameid']=='ESPORTSTMNT01/1030526'].head(10)

,date,patch,gameid,side,ban
0,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Blue,Yasuo
1,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Red,Cassiopeia
16120,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Blue,Akali
16121,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Red,Aatrox
32300,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Blue,Lucian
32301,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Red,Galio
48476,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Blue,Ezreal
48477,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Red,Sejuani
64668,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Blue,Vel'Koz
64669,2019-01-12 14:56:22,9.01,ESPORTSTMNT01/1030526,Red,Karthus


In [5]:
import duckdb

duckdb_conn = duckdb.connect(database=':memory:')
duckdb_conn.execute(f"CREATE TABLE pro_matches_summary AS SELECT * FROM read_parquet('{SILVER_DATASETS_DIR}/pro_matches_summary.parquet')")
duckdb_conn.execute("SELECT year(date) as match_year, COUNT(*) as match_count FROM pro_matches_summary GROUP BY match_year ORDER BY match_year").fetchall()

[(2019, 81250),
 (2020, 97470),
 (2021, 123020),
 (2022, 125260),
 (2023, 111060),
 (2024, 101920),
 (2025, 100380),
 (2026, 59820)]

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from metadata import metadata
import matches

metadata.create_last_match_handling_table()
dt = matches.x_get_last_match_date('pro')
print(f"Last match date for 'pro': {dt}")
metadata.update_last_match_date('pro', dt)
print(f"Last match date for 'pro': {metadata.get_last_match_date('pro')}")

Last match date for 'pro': 2026-07-02 00:18:31
Last match date for 'pro': 2026-07-02 00:18:31


# Get  Solo Queue matches

In [29]:


#check on json structure
sq = pd.read_json(BRONZE_DATASETS_DIR / 'riot_api' / '2026-07-20_match_info_results.json', lines=True)
sq.head(5)

,metadata,info
0,"{'dataVersion': '2', 'matchId': 'NA1_546386260...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
1,"{'dataVersion': '2', 'matchId': 'NA1_546382565...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
2,"{'dataVersion': '2', 'matchId': 'NA1_547805591...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
3,"{'dataVersion': '2', 'matchId': 'NA1_547803747...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
4,"{'dataVersion': '2', 'matchId': 'NA1_547796833...","{'endOfGameResult': 'GameComplete', 'gameCreat..."


In [30]:
# sq_info = sq.iloc[0].info
# df = pd.DataFrame(sq_info)
# df = pd.DataFrame([sq_info])
# df = pd.json_normalize(sq_info)
# df = pd.json_normalize(sq.iloc[0])
# df = pd.DataFrame(sq.iloc[0])
# df = pd.DataFrame([sq.iloc[0]])
# df = pd.json_normalize(sq)

# base_df = pd.DataFrame({
#     'match_id' : sq['metadata'].apply(lambda x: x['matchId']),
#     'match_info' : sq['info'].apply(lambda x: x)
# })

# base_df.head(5)

# for match in base_df.itertuples():
#     match_id = match.match_id
#     match_info = match.match_info
#     df = pd.json_normalize(match_info)
#     df['match_id'] = match_id
#     if 'matches_df' not in locals():
#         matches_df = df
#     else:
#         matches_df = pd.concat([matches_df, df], ignore_index=True)

match_ids = sq['metadata'].map(lambda x: x['matchId'])
matches_df = pd.json_normalize(sq['info'].tolist(), sep = "_")
matches_df.insert(0, 'match_id', match_ids)
matches_df.to_csv(SILVER_DATASETS_DIR / 'solo_queue_matches.csv', index=False)

In [46]:
matches_df.columns
#matches_df.head(5)
# participants = pd.json_normalize(matches_df.iloc[0]['participants'])
# participants.head(5)

# teams = pd.json_normalize(matches_df.iloc[0]['teams'])
# teams.head(5)




Index(['match_id', 'endOfGameResult', 'gameCreation', 'gameDuration',
       'gameEndTimestamp', 'gameId', 'gameMode', 'gameName',
       'gameStartTimestamp', 'gameType', 'gameVersion', 'mapId',
       'participants', 'platformId', 'queueId', 'teams', 'tournamentCode'],
      dtype='str')

In [ ]:
## Banned Champions List

teams = pd.json_normalize(matches_df.iloc[0]['teams'])
teams.head(5)

bans = pd.json_normalize(teams.iloc[0]['bans'])
banned_champions_id_list = bans['championId'].tolist()
print(f"Banned Champions ID List: {banned_champions_id_list}")


Banned Champions ID List: [107, 51, 201, 887, 134]


In [64]:
## List of Champions in the Match

participants = pd.json_normalize(matches_df.iloc[0]['participants'])
selected_champions = participants['championName'].tolist()
print(f"Selected Champions List: {selected_champions}")


Selected Champions List: ['Renekton', 'Evelynn', 'Heimerdinger', 'Kaisa', 'Janna', 'Gangplank', 'Nunu', 'TwistedFate', 'Jhin', 'Pyke']


In [ ]:
## Complete Champion Info

participants = pd.json_normalize(matches_df.iloc[1]['participants'])

columns = ['championName',
           'teamId', ##Side 100 - Blue / 200 - Red
           'teamPosition',
           'win', #Result
           'gameEndedInSurrender',
           'kills','deaths','assists',
           'totalDamageDealtToChampions', 'totalDamageTaken',
           'doubleKills','tripleKills','quadraKills','pentaKills',
           'longestTimeSpentLiving', 'largestKillingSpree', 'largestMultiKill',
           'totalMinionsKilled'
           ]

champions_info = participants[columns]
champions_info['teamKills'] = champions_info.groupby('teamId')['kills'].transform('sum')
champions_info['teamDeaths'] = champions_info.groupby('teamId')['deaths'].transform('sum')
champions_info['teamAssists'] = champions_info.groupby('teamId')['assists'].transform('sum')

champions_info.head(10)

,championName,teamId,teamPosition,win,gameEndedInSurrender,kills,deaths,assists,totalDamageDealtToChampions,totalDamageTaken,...,tripleKills,quadraKills,pentaKills,longestTimeSpentLiving,largestKillingSpree,largestMultiKill,totalMinionsKilled,teamKills,teamDeaths,teamAssists
0,Aatrox,100,TOP,True,False,19,2,14,42035,41077,...,1,0,0,1434,19,3,204,55,24,87
1,Jayce,100,JUNGLE,True,False,14,4,15,24546,18826,...,0,0,0,779,9,2,47,55,24,87
2,Velkoz,100,MIDDLE,True,False,7,8,12,23698,20637,...,0,0,0,239,2,2,166,55,24,87
3,MissFortune,100,BOTTOM,True,False,11,6,23,37080,17551,...,1,0,0,517,8,3,151,55,24,87
4,Nautilus,100,UTILITY,True,False,4,4,23,14838,20589,...,0,0,0,912,2,1,31,55,24,87
5,DrMundo,200,TOP,False,False,1,10,6,19057,44893,...,0,0,0,234,0,1,161,24,55,40
6,Sylas,200,JUNGLE,False,False,3,13,9,14134,43034,...,0,0,0,332,0,1,6,24,55,40
7,Akshan,200,MIDDLE,False,False,8,9,10,23009,23505,...,0,0,0,430,2,2,189,24,55,40
8,Samira,200,BOTTOM,False,False,9,11,6,30007,25663,...,1,1,0,256,2,4,218,24,55,40
9,Rell,200,UTILITY,False,False,3,12,9,9307,29804,...,0,0,0,219,0,1,25,24,55,40


In [63]:
## Match Basic Information
patch = '.'.join(matches_df.iloc[0]['gameVersion'].split('.')[:2])
print(f"Match Patch Version: {patch}")

gameCreationTime = pd.to_datetime(matches_df.iloc[0]['gameCreation'], unit='ms')
print(f"Match Creation Time: {gameCreationTime}")

gameDuration = pd.to_timedelta(matches_df.iloc[0]['gameDuration'], unit='s')
print(f"Match Duration: {gameDuration}")

gameResult = matches_df.iloc[0]['endOfGameResult']
print(f"Match Result: {gameResult}")

Match Patch Version: 16.1
Match Creation Time: 2026-01-14 02:42:34.424000
Match Duration: 0 days 00:33:34
Match Result: GameComplete


In [13]:
#Get list of files

start_date = pd.Timestamp('2026-07-20 12:15:30')
print(f"Start Date: {start_date}")

files_list = sorted([f for f in os.listdir(BRONZE_DATASETS_DIR / 'riot_api') 
                     if f.endswith('_match_info_results.json')
                     and f.split('_')[0] >= start_date.strftime('%Y-%m-%d')])
print(f"Files List: {files_list}")

Start Date: 2026-07-20 12:15:30
Files List: ['2026-07-20_match_info_results.json']


In [14]:
# Test the ingestion

base = pd.read_json(BRONZE_DATASETS_DIR / 'riot_api' / '2026-07-20_match_info_results.json', lines=True)
base_ids = base['metadata'].map(lambda x: x['matchId'])
base_df = pd.json_normalize(base['info'].tolist(), sep = "_")
base_df.insert(0, 'match_id', base_ids)

base_df.head(5)

,match_id,endOfGameResult,gameCreation,gameDuration,gameEndTimestamp,gameId,gameMode,gameName,gameStartTimestamp,gameType,gameVersion,mapId,participants,platformId,queueId,teams,tournamentCode
0,NA1_5463862600,GameComplete,1768358554424,2014,1768360683236,5463862600,CLASSIC,teambuilder-match-5463862600,1768358669686,MATCHED_GAME,16.1.737.4870,11,"[{'PlayerScore0': 0, 'PlayerScore1': 0, 'Playe...",NA1,420,"[{'bans': [{'championId': 107, 'pickTurn': 1},...",
1,NA1_5463825654,GameComplete,1768356695684,1557,1768358273771,5463825654,CLASSIC,teambuilder-match-5463825654,1768356717004,MATCHED_GAME,16.1.737.4870,11,"[{'PlayerScore0': 0, 'PlayerScore1': 0, 'Playe...",NA1,420,"[{'bans': [{'championId': 35, 'pickTurn': 1}, ...",
2,NA1_5478055910,GameComplete,1769677700799,2227,1769679998917,5478055910,CLASSIC,teambuilder-match-5478055910,1769677771851,MATCHED_GAME,16.2.741.3171,11,"[{'PlayerScore0': 0, 'PlayerScore1': 0, 'Playe...",NA1,420,"[{'bans': [{'championId': 35, 'pickTurn': 1}, ...",
3,NA1_5478037473,GameComplete,1769675185424,2013,1769677225104,5478037473,CLASSIC,teambuilder-match-5478037473,1769675212367,MATCHED_GAME,16.2.741.3171,11,"[{'PlayerScore0': 0, 'PlayerScore1': 0, 'Playe...",NA1,420,"[{'bans': [{'championId': 10, 'pickTurn': 1}, ...",
4,NA1_5477968337,GameComplete,1769667898633,1523,1769669436568,5477968337,CLASSIC,teambuilder-match-5477968337,1769667913867,MATCHED_GAME,16.2.741.3171,11,"[{'PlayerScore0': 0, 'PlayerScore1': 0, 'Playe...",NA1,420,"[{'bans': [{'championId': 234, 'pickTurn': 1},...",


In [9]:
ban_list_columns = ['date', 'patch', 'gameid', 'side', 'ban']
ban_list_df = pd.DataFrame(columns=ban_list_columns)


x = base_df.loc[:, ['match_id','gameStartTimestamp','gameVersion','teams']]
y = x.head(1).explode('teams').reset_index(drop=True).assign(
    #bans=lambda x: x['teams'].map(lambda y: y['bans'])
    date=lambda x: pd.to_datetime(x['gameStartTimestamp'], unit='ms'),
    patch=lambda x: x['gameVersion'].map(lambda y: '.'.join(y.split('.')[:2])),
    gameid=lambda x: x['match_id'],
    side=lambda x: x['teams'].map(lambda y: 'Blue' if y['teamId'] == 100 else 'Red'),
    bans=lambda x: x['teams'].map(lambda y: [str(b['championId']) for b in y['bans']]),
).drop(columns=['teams','gameStartTimestamp','gameVersion','match_id'])
# y.head(10)
#z = y['bans'].apply(lambda x: [b['championId'] for b in x] if isinstance(x, list) else [])
# z = y.explode('bans').reset_index(drop=True).dropna(subset=['bans'])
# z.head(10)

y_bans = (y['bans'].apply(lambda x: x if isinstance(x,list) else [])
                   .apply(lambda x: pd.Series(x[:], index=[f'ban{i+1}' for i in range(0,5)])))
y = pd.concat([y.drop(columns=['bans']), y_bans], axis=1)
y.head(10)

# id_cols = ['date','patch','gameid','side']
# ban_cols = [f'ban{i+1}' for i in range(0,5)]

# bans_melted = (
#     y.melt(
#         id_vars=id_cols,
#         value_vars=ban_cols,
#         var_name='ban_number',
#         value_name='bans'
#     ).drop(columns=['ban_number']).dropna(subset=['bans'])
# )

# #bans_melted.head(10)


# champ_dim = pd.read_parquet(SILVER_DATASETS_DIR / 'champions_dimension.parquet', engine='pyarrow')

# # print(z['bans'].dtype)
# # print(z['bans'].head())
# # print(champ_dim['championId'].dtype)
# # print(champ_dim['championId'].head())

# # w = (z.merge(champ_dim, how='left', left_on='bans', right_on='championId')
# #      .drop(columns=['championId', 'bans']).rename(columns={'championName': 'ban'}))

# bans_melted = (bans_melted.merge(champ_dim, how='left', left_on='bans', right_on='championId')
#      .drop(columns=['championId', 'bans']).rename(columns={'championName': 'ban'}).dropna(subset=['ban']))

# bans_melted.head(10)

# # id_cols = ['date','patch','gameid','side']
# # w_norm = (
# #     w.pivot_table(
# #         index = id_cols,
# #         columns = [ban for ban in ['ban1','ban2','ban3','ban4','ban5']],
# #         values='ban',
# #         aggfunc='first'
# #     )
# # )

# # print(w.head(5))



,date,patch,gameid,side,ban1,ban2,ban3,ban4,ban5
0,2026-01-14 02:44:29.686,16.1,NA1_5463862600,Blue,107,51,201,887,134
1,2026-01-14 02:44:29.686,16.1,NA1_5463862600,Red,950,-1,887,236,11


In [146]:
#Instead of using DDragon, I can create my own champions dimension from the matches data

x = base_df.loc[:, ['participants']]
y = x.explode('participants').reset_index(drop=True).assign(
    championId=lambda x: x['participants'].map(lambda y: str(y['championId']) if isinstance(y, dict) and 'championId' in y else None),
    championName=lambda x: x['participants'].map(lambda y: y['championName'] if isinstance(y, dict) and 'championName' in y else None)
).drop_duplicates(subset=['championId','championName'], keep='first').drop(columns=['participants']).dropna()
print(y.count())
y.head(10)

y.to_parquet(SILVER_DATASETS_DIR / 'champions_dimension.parquet', engine='pyarrow', index=False)


championId      173
championName    173
dtype: int64


In [ ]:
##Matches DF

champion_info_columns = ['championName', 'teamId', 'teamPosition', 'win', 'gameEndedInSurrender',
                         'kills','deaths','assists',
                         'totalDamageDealtToChampions', 'totalDamageTaken',
                         'doubleKills','tripleKills','quadraKills','pentaKills',
                         'longestTimeSpentLiving', 'largestKillingSpree', 'largestMultiKill',
                         'totalMinionsKilled']
match_info_columns = ['match_id', 'gameVersion', 'gameCreation', 'gameDuration', 'participants']

match_df = (base_df.loc[base_df['endOfGameResult'] == 'GameComplete', match_info_columns]
            .assign(
                patch=lambda x: x['gameVersion'].map(lambda y: '.'.join(y.split('.')[:2])),
                gameCreationTime=lambda x: pd.to_datetime(x['gameCreation'], unit='ms'),
                gameDurationTime=lambda x: pd.to_timedelta(x['gameDuration'], unit='s')
            )
            .drop(columns=['gameVersion','gameCreation','gameDuration'])
            .explode('participants').reset_index(drop=True)
)



champion_info_df = pd.json_normalize(match_df['participants']).loc[:, champion_info_columns].copy()
champion_info_df = (pd.concat([match_df, champion_info_df], axis=1, ignore_index=False)
                    .assign(
                        teamKills=lambda x: x.groupby(['match_id', 'teamId'])['kills'].transform('sum'),
                        teamDeaths=lambda x: x.groupby(['match_id', 'teamId'])['deaths'].transform('sum'),
                        side = lambda x: x['teamId'].map(lambda y: 'Blue' if y == 100 else 'Red')
                    )
                    .drop(columns=['participants','teamId'])
)

champion_info_df.head(100)


,match_id,patch,gameCreationTime,gameDurationTime,championName,teamPosition,win,gameEndedInSurrender,kills,deaths,...,tripleKills,quadraKills,pentaKills,longestTimeSpentLiving,largestKillingSpree,largestMultiKill,totalMinionsKilled,teamKills,teamDeaths,side
0,NA1_5463862600,16.1,2026-01-14 02:42:34.424,0 days 00:33:34,Renekton,TOP,True,False,11,6,...,0,0,0,602,5,2,212,48,34,Blue
1,NA1_5463862600,16.1,2026-01-14 02:42:34.424,0 days 00:33:34,Evelynn,JUNGLE,True,False,21,4,...,0,0,0,941,12,2,21,48,34,Blue
2,NA1_5463862600,16.1,2026-01-14 02:42:34.424,0 days 00:33:34,Heimerdinger,MIDDLE,True,False,3,9,...,0,0,0,547,0,1,210,48,34,Blue
3,NA1_5463862600,16.1,2026-01-14 02:42:34.424,0 days 00:33:34,Kaisa,BOTTOM,True,False,11,8,...,0,0,0,521,3,2,212,48,34,Blue
4,NA1_5463862600,16.1,2026-01-14 02:42:34.424,0 days 00:33:34,Janna,UTILITY,True,False,2,7,...,0,0,0,409,0,1,27,48,34,Blue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NA1_5477782047,16.2,2026-01-29 03:20:57.354,0 days 00:31:05,Jayce,TOP,True,True,10,7,...,0,0,0,483,3,1,234,31,20,Red
96,NA1_5477782047,16.2,2026-01-29 03:20:57.354,0 days 00:31:05,DrMundo,JUNGLE,True,True,7,4,...,0,0,0,381,7,2,40,31,20,Red
97,NA1_5477782047,16.2,2026-01-29 03:20:57.354,0 days 00:31:05,Viktor,MIDDLE,True,True,8,2,...,0,0,0,829,5,2,290,31,20,Red
98,NA1_5477782047,16.2,2026-01-29 03:20:57.354,0 days 00:31:05,Hwei,BOTTOM,True,True,4,3,...,0,0,0,638,4,1,223,31,20,Red


In [ ]:
# Result review

from utils import SOLOQ_MATCHES_BAN_LIST_FILE, SOLOQ_MATCHES_SUMMARY_FILE, CHAMPIONS_DIMENSION_FILE

# ban_list = pd.read_parquet(SOLOQ_MATCHES_BAN_LIST_FILE)
# ban_list.loc[ban_list['gameid']== 'NA1_5463862600'].head(20)

# matches = pd.read_parquet(SOLOQ_MATCHES_SUMMARY_FILE)
# #matches[['champion', 'champion_id']].head(10)
# matches.loc[matches['match_id']== 'NA1_5463862600',['champion','kills','teamkills']].head(20)

dim = pd.read_parquet(CHAMPIONS_DIMENSION_FILE)
duplicate_champions = dim[dim.duplicated(subset=['champion'], keep=False)] \
    .sort_values('champion')
duplicate_champions.head(10)



,champion,champion_id
